In [1]:
import sys
sys.path.append("../qfb_optimization/")
sys.path.append("..")
# Add REGCOIL executable to path before importing replicate_lgradb
sys.path.append("~/regcoil")

import latexplot
latexplot.set_cmap(4)
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import simsopt
import joblib
from joblib import Memory
from scipy.spatial.distance import cdist
from scipy.stats import linregress
from pathlib import Path
import pandas as pd
from simsopt.configs import get_QUASR_data
from simsopt.mhd import vmec_diagnostics
from vmecpp.simsopt_compat import Vmec
from replicate_lgradb.find_single_l import find_regcoil_distance


SINGLE_STAGE_PATH = Path.home() / "single-stage-opt/"


def coil_surf_distance(curves, lcfs) -> np.ndarray:
    pointcloud1 = lcfs.gamma().reshape((-1, 3))
    distances = [np.min(cdist(pointcloud1, c.gamma()), axis=0) for c in curves]
    return np.array(distances).T


def compute_coil_surf_dist(simsopt_filename):
    surfaces, coils = simsopt.load(simsopt_filename)
    lcfs = surfaces[-1].to_RZFourier()

    curves = [c.curve for c in coils]
    return coil_surf_distance(curves, lcfs)

def vs_plot(x_data, y_data, labels=None):
    x_vals, x_label = x_data
    y_vals, y_label = y_data
    title = x_label + " vs " + y_label

    if len(np.shape(y_vals)) >= 2:
        y_vals = np.array(y_vals).T
    elif len(np.shape(y_vals)) == 1:
        y_vals = np.reshape(y_vals, (1,) + np.shape(y_vals))

    assert len(x_vals) == len(y_vals[0])
    if labels is not None:
        assert len(labels) == len(y_vals), f"{len(labels)} != {np.shape(y_vals)}"

    # Linear fit
    # TODO this Fails because some values are inf!!
    for i, y in enumerate(y_vals):
        plt.scatter(x_vals, y, label=title if labels is None else labels[i], s=4)
        reg = linregress(x_vals, y)
        plt.axline(
            xy1=(0, reg.intercept),
            slope=reg.slope,
            color="k" if len(y_vals) == 1 else plt.rcParams["axes.prop_cycle"].by_key()["color"][i],
            label=f"Linear fit {i}: $R^2$ = {reg.rvalue**2:.3f}",
        )

    plt.xlabel(x_label)
    plt.ylabel(y_label)
    plt.title(title)
    if np.min(y_vals) >= 0:
        plt.gca().set_ylim(bottom=0)
    plt.gca().set_xlim(left=0)
    plt.grid(True)
    plt.legend()


In [2]:
df:pd.DataFrame = pd.read_pickle("../quasr_exploration/QUASR_db/QUASR_08072024.pkl")

# Select the subset for analysis. In the thesis we present results using two different filters. 
# A dedicated copy of the script with exact parameters used to generate the clustering results is in the plots/cluster_by_coils/ folder.

# Filter df by constant number of coils
df["total_coils"] = df["nc_per_hp"] * df["nfp"]
df = df[df["total_coils"] == 24]
# df[df["total_coils"] == 24]["nfp"].unique()
# df[df["total_coils"] == 20]["nfp"].unique() # Bigger dataset
# df = df.sample(n=60, replace=False, random_state=42)

In [4]:
ids = df["ID"].tolist()
print(len(ids), "configurations to process.")
parallel = joblib.parallel.Parallel(backend="threading", return_as="generator", n_jobs=32)
parallel_download_gen = parallel(joblib.parallel.delayed(get_QUASR_data)(idx) for idx in ids)
valid_results_gen = filter(lambda nested: nested[1][1] is not None, zip(ids, parallel_download_gen))
# Flatten the nested tuple 
results = [(nested[0], *nested[1]) for nested in valid_results_gen]

1624 configurations to process.
ID=1970266 is cached, loading...
ID=1970237 is cached, loading...
ID=1970249 is cached, loading...
ID=1970231 is cached, loading...
ID=1970260 is cached, loading...
ID=1970264 is cached, loading...
ID=1970288 is cached, loading...
ID=1970300 is cached, loading...
ID=1970256 is cached, loading...
ID=1970262 is cached, loading...
ID=1970263 is cached, loading...
ID=1970292 is cached, loading...
ID=1970317 is cached, loading...
ID=1970283 is cached, loading...
ID=1970250 is cached, loading...
ID=1970253 is cached, loading...
ID=1970251 is cached, loading...
ID=1970279 is cached, loading...
ID=1970316 is cached, loading...
ID=1970276 is cached, loading...
ID=1970245 is cached, loading...
ID=1970248 is cached, loading...
ID=1970259 is cached, loading...
ID=1970287 is cached, loading...
ID=1970298 is cached, loading...
ID=1970299 is cached, loading...
ID=1970289 is cached, loading...
ID=1970295 is cached, loading...
ID=1970274 is cached, loading...
ID=1970228 

In [5]:
def normalize_scale(surface, constant_minor=True):
  # Scaling factor for either constant minor or major radius
  if constant_minor:
    scaling = 1.704 / surface.minor_radius()
  else: 
    scaling = 1.0 / surface.major_radius()
  return scaling

In [7]:

for idx, surfs, coils in results:
    # The distances here were verified with the QUASR database GUI and are correct.
    # simsopt_path = f"{SINGLE_STAGE_PATH}/quasr_exploration/QUASR_db/simsopt_serials/{simsopt_name[6:10]}/{simsopt_name}"
    cs_dist = coil_surf_distance([c.curve for c in coils], surfs[-1]) * normalize_scale(surfs[-1])
    df.loc[df['ID'] == idx, 'coil_surf_distance_mean'] = np.mean(cs_dist)
    df.loc[df['ID'] == idx, 'coil_surf_distance_min'] = np.min(cs_dist)
df[['coil_surf_distance_min', 'coil_surf_distance_mean']]

,coil_surf_distance_min,coil_surf_distance_mean
0,6.134034,43.114138
1,6.133678,42.872273
2,6.133173,38.487907
3,6.134328,43.411922
4,6.134355,43.545392
...,...,...
3,1.533991,10.269600
4,1.533595,10.399354
5,1.533563,10.373176
6,1.533707,10.231859


In [ ]:
import pandas as pd
pd.options.plotting.backend = "plotly"

In [73]:
# Create a new dataframe with an (approximately) uniform distribution in coil_surf_distance_min
col = "coil_surf_distance_min"

# Parameters: adjust desired_total and n_bins as you like
n_bins = min(100, len(df))  # number of bins along the distance axis

# target samples per bin (at least 1)
target_per_bin = 100

# Bin the data uniformly in value space and sample up to target_per_bin from each bin
bins = np.linspace(df[col].min(), df[col].max(), n_bins + 1)
df["_bin"] = pd.cut(df[col], bins=bins, include_lowest=True)
df["coil_surf_distance_min"].hist(bins=n_bins)
# Randomly drop samples from bins that have >100 entries until each has 100
rng_gen = np.random.default_rng(42)  # use existing rng for reproducibility

# Identify bins with more than 100 entries
bin_counts = df["_bin"].value_counts()
over_represented = bin_counts[bin_counts > 100]
print("Bins over 100 before drop:\n", over_represented)

resampled_rows = []
for b, cnt in over_represented.items():
    n_drop = int(cnt - target_per_bin)
    if n_drop <= 0:
        continue
    resampled_rows.append(
        df[df["_bin"] == b].sample(
            n=100, replace=False, random_state=42
        )
    )
correctly_represented_df = df[~df["_bin"].isin(over_represented.index)]
uniform_df = pd.concat([correctly_represented_df] + resampled_rows)
# Confirm result
uniform_df["_bin"].value_counts()

Bins over 100 before drop:
 _bin
(6.128, 6.189]    996
(3.045, 3.105]    372
(2.017, 2.078]    109
Name: count, dtype: int64


_bin
(3.045, 3.105]                 100
(2.017, 2.078]                 100
(6.128, 6.189]                 100
(1.5330000000000001, 1.594]     16
(6.249, 6.31]                   14
                              ... 
(5.705, 5.766]                   0
(5.947, 6.007]                   0
(5.886, 5.947]                   0
(7.337, 7.398]                   0
(7.458, 7.519]                   0
Name: count, Length: 100, dtype: int64

In [75]:
uniform_df["coil_surf_distance_min"].hist(bins=n_bins)

In [76]:
location = "./.cachedir"
memory = Memory(location, verbose=0)


def lgradbsc(computed) -> dict:
    gradB = np.array(
        [
            [computed.grad_B__XX, computed.grad_B__YX, computed.grad_B__ZX],
            [computed.grad_B__XY, computed.grad_B__YY, computed.grad_B__ZY],
            [computed.grad_B__XZ, computed.grad_B__YZ, computed.grad_B__ZZ],
        ]
    )
    scalarGradB = np.sqrt(gradB[0, 0] ** 2 + gradB[1, 1] ** 2 + gradB[2, 2] ** 2)
    LgradBs = np.sqrt(2) * computed.modB / np.linalg.norm(gradB, ord="fro", axis=(0, 1))
    LgradBscalar = computed.modB / scalarGradB
    # "$L_{\\max \\sigma}$"
    LgradB2maxsigma = computed.modB / np.linalg.norm(gradB, ord=2, axis=(0, 1))
    # "$L_{\\sigma}$"
    # TODO: Not entirely sure if the authors meant the nuclear norm (sum of singular values) or this norm (root mean square of singular values)
    LgradB2sigma = (
        np.sqrt(2)
        * computed.modB.flatten()
        # / np.linalg.norm(gradB, ord="nuc", axis=(0, 1))
        / np.sqrt(
            np.sum(
                np.linalg.svd(gradB.reshape((3, 3, -1)).T, compute_uv=False) ** 2,
                axis=1,
            )
        )
    )
    Lgradbgradb = (
        computed.modB
        * computed.modB
        / np.linalg.norm(
            np.sum(
                np.array([computed.B_X, computed.B_Y, computed.B_Z]) * gradB, axis=0
            ),
            axis=0,
        )
    )

    def fsa(sqrtg, thing):
        return np.sum(sqrtg * thing) / np.sum(sqrtg)

    sqrtg = np.ascontiguousarray((computed.sqrt_g_vmec).squeeze())
    Bmod = np.ascontiguousarray((computed.modB).squeeze())
    fsa_B = fsa(sqrtg.T.squeeze(), Bmod.T.squeeze())  # Flux surface averaged B
    fsa_B_metric = np.ascontiguousarray(
        (fsa_B * np.sqrt(2)) / (computed.norm_grad_B.squeeze().T * computed.L_reference)
    )[:, :, None]
    return {
        "$L^*_{\\nabla \\vec{B}}$": np.min(LgradBs),
        "$L_{\\vec{B} \\cdot \\nabla \\vec{B}}$": np.min(Lgradbgradb),
        "$L_{\\nabla |B|}$": np.min(LgradBscalar),
        "$L_{\\max \\sigma}$": np.min(LgradB2maxsigma),
        "$L_{\\sigma}$": np.min(LgradB2sigma),
        "$L_{fsa_B}$": np.min(fsa_B_metric),
    }
    return np.array(
        [
            np.min(LgradBs),
            np.min(Lgradbgradb),
            np.min(LgradBscalar),
            np.min(LgradB2maxsigma),
            np.min(LgradB2sigma),
            np.min(fsa_B_metric),
        ]
    )


@memory.cache(ignore=["vmec"])
def vmec_lgradbsc(vmec: Vmec, cache_invalidator: int | str):
    # try:
    #     vmec.run(max_threads=4)
    # except Exception as e:
    #     print(f"Error! {str(e)}")
    #     return np.zeros(6)
    s = [1]
    ntheta = 128
    nphi = 128
    theta = np.linspace(0, 2 * np.pi, ntheta)
    phi = np.linspace(0, 2 * np.pi / vmec.boundary.nfp, nphi)
    # data = vmec_diagnostics.vmec_compute_geometry(vmec_diagnostics.vmec_splines(vmec), s, theta, phi)
    # B0 = np.mean(data.modB)
    # vmec.indata.phiedge = 5.865 * vmec.indata.phiedge / B0
    vmec.recompute_bell()
    try:
        vmec.run(max_threads=4)
        computed = vmec_diagnostics.vmec_compute_geometry(
            vmec_diagnostics.vmec_splines(vmec), s, theta, phi
        )
        return lgradbsc(computed)
    except Exception as e:
        print(f"VMEC didn't converge on the second attempt! {str(e)}")
    return {
        "$L^*_{\\nabla \\vec{B}}$": 0.0,
        "$L_{\\vec{B} \\cdot \\nabla \\vec{B}}$": 0.0,
        "$L_{\\nabla |B|}$": 0.0,
        "$L_{\\max \\sigma}$": 0.0,
        "$L_{\\sigma}$": 0.0,
        "$L_{fsa_B}$": 0.0,
    }
    return np.zeros(6)

In [ ]:
from simsopt import geo
lgradb_variants = {}
regcoil_distances = {}
num=0

def one_config(idx, surfs, coils):
    global num
    assert df['ID'].isin([idx]).any()
    num += 1
    s : geo.SurfaceRZFourier = surfs[-1].to_RZFourier().copy()

    # If the configurations are all scaled to the same major radius instead of minor radius, the results that follow are not qualitatively different
    scaling = normalize_scale(s, constant_minor=True)
    s.rc *= scaling
    s.zs *= scaling
    s.recompute_bell()
    vmec = Vmec("input.preset", keep_all_files=True, verbose=False)
    assert vmec.indata is not None
    vmec.indata.nfp = vmec.boundary.nfp
    vmec.boundary = s
    lgradb_variants[idx] = vmec_lgradbsc(vmec, idx)
    print(num, df[df['ID'] == idx]["coil_surf_distance_min"], ": R0:", s.major_radius(), s.minor_radius(), "LgradB:", lgradb_variants[idx])
    assert len(lgradb_variants[idx]) == 6

    # Add LgradB variants to DataFrame
    for name, value in lgradb_variants[idx].items():
        df.loc[df['ID'] == idx, name] = value

    # if np.allclose(np.array([value for value in lgradb_variants[idx].values()]), 0): 
    #     regcoil_distances[f"{idx:07d}"] = 0
    # else:
    #     # Must pass 
    #     vmec.run()
    #     regcoil_distances[f"{idx:07d}"] = find_regcoil_distance(vmec, idx)
    # print("Lregcoil", regcoil_distances[f"{idx:07d}"])
    
    # Add REGCOIL distance to DataFrame
    # df.loc[df['ID'] == idx, 'L_REGCOIL'] = regcoil_distances[f"{idx:07d}"]


for idx, surfs, coils in results:
    one_config(idx, surfs, coils)

1 0    6.134034
Name: coil_surf_distance_min, dtype: float64 : R0: 40.89353964656943 1.7040000000000026 LgradB: {'$L^*_{\\nabla \\vec{B}}$': np.float64(13.640942992067336), '$L_{\\vec{B} \\cdot \\nabla \\vec{B}}$': np.float64(15.690704017302876), '$L_{\\nabla |B|}$': np.float64(10.849811254512462), '$L_{\\max \\sigma}$': np.float64(11.900069287143905), '$L_{\\sigma}$': np.float64(13.640942992067334), '$L_{fsa_B}$': np.float64(8.215873145585888)}
2 1    6.133678
Name: coil_surf_distance_min, dtype: float64 : R0: 40.895000306733685 1.7040000000000077 LgradB: {'$L^*_{\\nabla \\vec{B}}$': np.float64(17.439765515485238), '$L_{\\vec{B} \\cdot \\nabla \\vec{B}}$': np.float64(20.297505701635114), '$L_{\\nabla |B|}$': np.float64(18.18547300956109), '$L_{\\max \\sigma}$': np.float64(15.421884836573692), '$L_{\\sigma}$': np.float64(17.439765515485238), '$L_{fsa_B}$': np.float64(10.383911528699638)}
3 2    6.133173
Name: coil_surf_distance_min, dtype: float64 : R0: 40.89548828626212 1.704000000000

/tmp/ipykernel_280226/2560079618.py:15: RuntimeWarning:

divide by zero encountered in divide



22 21    6.134392
Name: coil_surf_distance_min, dtype: float64 : R0: 40.89674676204559 1.7040184059336754 LgradB: {'$L^*_{\\nabla \\vec{B}}$': np.float64(12.784013623297444), '$L_{\\vec{B} \\cdot \\nabla \\vec{B}}$': np.float64(16.150877571381336), '$L_{\\nabla |B|}$': np.float64(10.378063191837343), '$L_{\\max \\sigma}$': np.float64(11.387034192028246), '$L_{\\sigma}$': np.float64(12.784013623297442), '$L_{fsa_B}$': np.float64(7.879615886179336)}


/tmp/ipykernel_280226/2560079618.py:15: RuntimeWarning:

divide by zero encountered in divide



23 22    6.134394
Name: coil_surf_distance_min, dtype: float64 : R0: 40.896843441127785 1.7040203514185501 LgradB: {'$L^*_{\\nabla \\vec{B}}$': np.float64(12.82699697194146), '$L_{\\vec{B} \\cdot \\nabla \\vec{B}}$': np.float64(16.277995156512937), '$L_{\\nabla |B|}$': np.float64(10.236589684723992), '$L_{\\max \\sigma}$': np.float64(11.502223619499823), '$L_{\\sigma}$': np.float64(12.826996971941456), '$L_{fsa_B}$': np.float64(7.8976853472728985)}


...

42 41    6.134125
Name: coil_surf_distance_min, dtype: float64 : R0: 40.89250103492647 1.7039441323858442 LgradB: {'$L^*_{\\nabla \\vec{B}}$': np.float64(12.683589719667573), '$L_{\\vec{B} \\cdot \\nabla \\vec{B}}$': np.float64(13.3703542285763), '$L_{\\nabla |B|}$': np.float64(10.432413695280315), '$L_{\\max \\sigma}$': np.float64(10.691883798303296), '$L_{\\sigma}$': np.float64(12.683589719667573), '$L_{fsa_B}$': np.float64(7.798574811060333)}


/tmp/ipykernel_280226/2560079618.py:15: RuntimeWarning:

divide by zero encountered in divide



43 42    6.133183
Name: coil_surf_distance_min, dtype: float64 : R0: 40.808215496087385 1.704 LgradB: {'$L^*_{\\nabla \\vec{B}}$': np.float64(14.741717566056588), '$L_{\\vec{B} \\cdot \\nabla \\vec{B}}$': np.float64(10.551750586693746), '$L_{\\nabla |B|}$': np.float64(11.44501212945284), '$L_{\\max \\sigma}$': np.float64(13.511684656824077), '$L_{\\sigma}$': np.float64(14.741717566056588), '$L_{fsa_B}$': np.float64(8.170118696184394)}


/tmp/ipykernel_280226/2560079618.py:15: RuntimeWarning:

divide by zero encountered in divide



44 43    6.133612
Name: coil_surf_distance_min, dtype: float64 : R0: 40.89413220464704 1.7039788202669424 LgradB: {'$L^*_{\\nabla \\vec{B}}$': np.float64(10.522860613074753), '$L_{\\vec{B} \\cdot \\nabla \\vec{B}}$': np.float64(8.643329737972401), '$L_{\\nabla |B|}$': np.float64(10.248031489771988), '$L_{\\max \\sigma}$': np.float64(8.251268368996758), '$L_{\\sigma}$': np.float64(10.522860613074753), '$L_{fsa_B}$': np.float64(6.620312695333677)}
VMEC didn't converge on the second attempt! Error while running VMEC++: VMEC++ did not converge
45 44    6.134399
Name: coil_surf_distance_min, dtype: float64 : R0: 41.24590019148358 1.7040000000000033 LgradB: {'$L^*_{\\nabla \\vec{B}}$': 0.0, '$L_{\\vec{B} \\cdot \\nabla \\vec{B}}$': 0.0, '$L_{\\nabla |B|}$': 0.0, '$L_{\\max \\sigma}$': 0.0, '$L_{\\sigma}$': 0.0, '$L_{fsa_B}$': 0.0}
45 44    6.134399
Name: coil_surf_distance_min, dtype: float64 : R0: 41.24590019148358 1.7040000000000033 LgradB: {'$L^*_{\\nabla \\vec{B}}$': 0.0, '$L_{\\vec{B} \

In [ ]:
a = set([f"{idx:07d}" for idx in lgradb_variants.keys()])
b = set(regcoil_distances.keys())
a == b

In [ ]:
def vs_plot2(x_data, y_data, labels=None):
    x_vals, x_label = x_data
    y_vals, y_label = y_data
    title = x_label + " vs " + y_label
    filename = title.replace(" ", "_") + ".csv"

    # Ensure y_vals has shape (n_series, n_points)
    if len(np.shape(y_vals)) >= 2:
        y_vals = np.array(y_vals).T
    elif len(np.shape(y_vals)) == 1:
        y_vals = np.reshape(y_vals, (1,) + np.shape(y_vals))

    assert len(x_vals) == len(y_vals[0])
    if labels is not None:
        assert len(labels) == len(y_vals), f"{len(labels)} != {np.shape(y_vals)}"

    # Build DataFrame
    plot_df = pd.DataFrame({x_label: x_vals})
    for i, y in enumerate(y_vals):
        col_name = labels[i] if labels is not None else f"{y_label}_{i}"
        plot_df[col_name] = y

    # Save to CSV
    plot_df.to_csv(filename, index=False)
    return plot_df  # return in case you want to use it in memory too

In [35]:
cmap10 = False
regcoil_plot = True
bdistrib_plot = False
latexplot.set_cmap(4)
if cmap10: 
    plt.rcParams['axes.prop_cycle'] = matplotlib.cycler(color=[plt.get_cmap('tab10')(e) for e in range(4)])

lgradB_names = ["$L^*_{\\nabla \\vec{B}}$", 
                "$L_{\\vec{B} \\cdot \\nabla \\vec{B}}$", 
                "$L_{\\nabla |B|}$", 
                "$L_{\\max \\sigma}$",
                "$L_{\\sigma}$",
                "$L_{fsab}$"
                ]
LgradB_keyed = {}
coil_surf_dist = {}

for idx, surfs, coils in results:
    filename =  f"{idx:07d}" 
    print("Loading", filename)
    if idx in lgradb_variants:
        computed = lgradb_variants[idx]
    else:
        # VMEC didn't converge for this configuration, so its not a valid result
        continue

    # $L_{REGCOIL}$
    simsopt_name = filename.replace("input.", "serial").replace(
        "_output.h5", ".json"
    )
    LgradB_keyed[simsopt_name] = lgradb_variants[idx]
    LgradB = LgradB_keyed[simsopt_name][0]

    # The distances here were verified with the QUASR database GUI and are correct.
    # simsopt_path = f"{SINGLE_STAGE_PATH}/quasr_exploration/QUASR_db/simsopt_serials/{simsopt_name[6:10]}/{simsopt_name}"
    coil_surf_dist[simsopt_name] = coil_surf_distance([c.curve for c in coils], surfs[-1]) * normalize_scale(surfs[-1], constant_minor=True) #compute_coil_surf_dist(simsopt_path)
    
    print(
        simsopt_name,
        "has minimum filament coil distance",
        np.min(coil_surf_dist[simsopt_name]),
    )

#########################

# Extract filenames and corresponding values for plotting
latexplot.set_cmap(len(lgradB_names))
if cmap10: 
    plt.rcParams['axes.prop_cycle'] = matplotlib.cycler(color=[plt.get_cmap('tab10')(e) for e in range(4)])
validlgradb_dict = {key: value for key, value in LgradB_keyed.items() if np.all(value > 0.0)} #
filenames = list(validlgradb_dict.keys())

bothLgradBandRegcoil = list(validlgradb_dict.keys())
if regcoil_plot:
    validr_dict = {key: value for key, value in regcoil_distances.items() if (value != 0) and np.isfinite(value)}
    bothLgradBandRegcoil = list(set(validlgradb_dict.keys()).intersection(validr_dict.keys()))
    regcoil_vals = (np.array([regcoil_distances[f] for f in bothLgradBandRegcoil]), "$L_{REGCOIL}$")
    
    LgradB_vals = (np.array([LgradB_keyed[f][:len(lgradB_names)] for f in bothLgradBandRegcoil]), "$L^*_{\\nabla B}$") 
    coil_min_vals = (
        [np.min(coil_surf_dist[f]) for f in bothLgradBandRegcoil],
        "QUASR coil distance",
    )
    # latexplot.figure()
    # vs_plot2(regcoil_vals, LgradB_vals, lgradB_names)
    # latexplot.savenshow("regcoil_vs_lgradb")
    # latexplot.figure()
    # vs_plot2(coil_min_vals, regcoil_vals)
    vs_plot2(coil_min_vals, LgradB_vals, lgradB_names)
    # latexplot.savenshow("regcoil_vs_quasr")



coil_min_vals = (
    np.array([np.min(coil_surf_dist[f]) for f in bothLgradBandRegcoil]),
    "QUASR coil distance",
)

# Do we want to plot the same points as for regcoil?
LgradB_vals = (np.array([LgradB_keyed[f] for f in filenames]), "$L^*_{\\nabla B}$") 
coil_min_vals = (
    np.array([np.min(coil_surf_dist[f]) for f in filenames]),
    "QUASR coil distance",
)

if cmap10: 
    plt.rcParams['axes.prop_cycle'] = matplotlib.cycler(color=[plt.get_cmap('tab10')(e) for e in range(len(lgradB_names))])

#########################
# latexplot.figure()
# vs_plot2(coil_min_vals, LgradB_vals, lgradB_names)
# latexplot.savenshow("lgradb_vs_quasr")

Loading 0650216
Loading 0191353
Loading 0373297
Loading 0772209
Loading 0222035
Loading 1055756
Loading 0804745
Loading 1913699
Loading 0050151
Loading 0190403
Loading 0646186
Loading 0654020
Loading 0051220
Loading 0896360
Loading 0219656
Loading 1057570
Loading 0614426
Loading 0369275
Loading 0226032
Loading 1053505
Loading 0226401
Loading 0651560
Loading 0645863
Loading 0925208
Loading 0022999
Loading 0805055
Loading 0892913
Loading 0802600
Loading 0895856
Loading 0054222
Loading 0652151
Loading 0621426
Loading 0805105
Loading 1451198
Loading 1301577
Loading 0616861
Loading 1084611
Loading 1089567
Loading 0892742
Loading 0368249
Loading 0803326
Loading 0650323
Loading 2447680
Loading 0219065
Loading 0187912
Loading 0223859
Loading 1086114
Loading 0898015
Loading 1842804
Loading 1050985
Loading 0051431
Loading 0893551
Loading 0923633
Loading 0894325
Loading 0621216
Loading 1451371
1451371 has minimum filament coil distance 4.606347763359004
Loading 0023191
0023191 has minimum filamen

In [ ]:
dfexport = pd.DataFrame(
  data=lgradb_variants.keys(),
  columns=["QUASR ID"]
)
for i, name in enumerate(lgradB_names):
  dfexport[name] = LgradB_vals[0][:,i]

dfexport["coil_min_distance"] = coil_min_vals[0]


std_deviations = []
for idx, surfs, coils in results
  arr = np.array([coil.current.get_value() for coil in coils])
  std_deviations.append(np.std(arr)/np.mean(np.abs(arr)))

dfexport["coil current std deviation"] = std_deviations
dfexport.to_csv("lgradb_on_quasr_kappel.csv", index=False)

In [ ]:
import pickle

with open("lgradb_vs_quasr2.pkl", "wb") as f:
    pickle.dump((coil_min_vals, LgradB_vals, lgradB_names), f)

vs_plot(coil_min_vals, LgradB_vals, lgradB_names)
plt.savefig("lgradb_vs_quasr2.png", dpi=300, bbox_inches="tight")

In [ ]:
# Get the data for some of the outliers from LgradB
df["$L_{\\nabla |B|}$"] = df["$L^*_{\\nabla B}$"].map(lambda x: 0 if np.any(np.isnan(x)) else x[2])
df[(df["$L_{\\nabla |B|}$"]<5) & (df["QUASR coil distance"]>4) & (df["QUASR coil distance"]<16)]

# Failiure cases

In [34]:
df.reset_index().to_csv("quasr_subset.csv")